# N058 · Top-K、频率与流式维护

**目标：** 用大小K的堆保留当前所需候选。

**先修：** N057, N020, N041。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 小根堆维护最大K项；频率Top-K；流式输入；K边界。

**配套讲解来源：** [同名逐章意见](../../comment/058_heap_top_k.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决三件密切相关的事：

1. 一个数组里"第 k 大"的数是多少？（不是要整个排序，只想知道那一个名次。）
2. 出现频率最高的 k 个数是哪几个？
3. 更难的版本：数是一个一个流过来的，每来一个新数，你都要立刻报出"当前的第 k 大"——不能回头重算。

具体到数字的小例子：

- 输入 `[3,2,1,5,6,4]`，k=2，输出 `5`。为什么？把数组降序排列是 6,5,4,3,2,1，排第 2 位的是 5，所以第 2 大是 5。
- 输入 `[1,1,1,2,2,3]`，k=2，输出 `{1,2}`。为什么？数一下：1 出现 3 次、2 出现 2 次、3 出现 1 次，出现最多的两个数就是 1 和 2。
- 流式例子（notebook 小实例原数据）：先初始化 KthLargest(3, [4,5,8,2])，然后依次 add 3、5、10、9、4，五次返回应该是 `[4,5,5,8,8]`。以最后一步为例：此时见过的数是 4,5,8,2,3,5,10,9,4，降序前三名是 10、9、8，第三大是 8，所以返回 8。

这三件事的共同点是：你不需要全部 n 个数的完整排名，只需要"前 k 名"这一小撮。

## 2. 基础知识：先读懂这些词

- **Top-K 问题**：只求最大的 k 个（或第 k 大）而不做完整排序的一类问题。n 很大、k 很小时，专门做 Top-K 比全排序便宜得多。
- **小根堆维护最大 K 项**：听起来拧巴但很好用——你想留"最大的 k 个"，那最该随时知道的是"这 k 个里最小的那个"（它是准入门槛），而小根堆的堆顶恰好就是它。堆里放 k 个数，堆顶即"当前第 k 大"。
- **堆顶门槛**：新来的数如果不大于堆顶，它连前 k 的门槛都够不着，直接扔掉；如果大于堆顶，它挤掉堆顶（堆顶出局），自己进堆。
- **heapreplace**：`heapq.heapreplace(heap, x)` 是"弹出堆顶并放入 x"的一步操作，比先 heappop 再 heappush 省一次树调整。本章代码用它实现"挤掉门槛"。
- **Counter（计数器）**：标准库 `collections.Counter`，一行代码把"每个数出现几次"数成字典。频率 Top-K 的第一步永远是先计数。
- **频次并列**：两个数出现次数一样时，题目答案就不唯一了。本章为了展示稳定，规定"频次相同取数值较小的那个"。
- **流式 / 在线**：数据不是一次性给你，而是一个一个来，每个时刻都要能回答查询。特点是"来一个、处理一个、随时答"，不能等收齐了再算。

## 3. 思路推导：从小例子开始

Step 1：先想清楚为什么"全排序"不划算。n=10⁶ 个数排一次序约 2×10⁷ 次比较；而只要前 k=10 大，用门槛堆扫一遍只需约 10⁶×log₂10≈3.3×10⁶ 次比较，便宜近一个数量级。

Step 2：维护一个大小不超过 k 的小根堆，扫描数组。前 k 个数直接进堆；之后每个新数 x 和堆顶比：x 大于堆顶就"换门槛"，否则跳过。扫完时，堆顶就是第 k 大，堆里那 k 个就是前 k 大。

Step 3：频率版先 Counter 计数，把"出现次数"当作新的权重做同样的 Top-K。注意答案的单位从"数值"变成了"不同的数（键）"。

Step 4：流式版把 Step 2 拆开：初始化时把前 k 个喂进堆，之后每 add 一个数就执行一遍"进堆 + 超员就弹堆顶 + 返回堆顶"。

手算演示（notebook 小实例的原数据）：`KthLargest(3, [4,5,8,2])` 初始化后，堆中保留 {4,5,8}（2 被挤掉了：2 进堆后堆有 4 个数，弹出最小值 2），堆顶是 4，也就是"当前第 3 大是 4"。之后每次 add：

| 新值 | 保留的 Top3 | 第3大 | 发生了什么 |
| --- | --- | --- | --- |
| 3 | [4,5,8] | 4 | 3 不大于堆顶 4，被门槛挡掉 |
| 5 | [5,5,8] | 5 | 5>4，挤掉 4 进堆（流里本来有个 5，所以有两个 5） |
| 10 | [5,8,10] | 5 | 10 挤掉堆顶 5 |
| 9 | [8,9,10] | 8 | 9 挤掉堆顶 5 |
| 4 | [8,9,10] | 8 | 4 不大于堆顶 8，被挡掉 |

这张表和 notebook 小实例生成的表格完全一致，你可以对照运行结果核对。

## 4. 核心代码：kth_largest_heap

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def kth_largest_heap(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('invalid rank')
    heap = []
    for x in nums:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return heap[0]
```

### 逐段读懂代码

### 1. `kth_largest_heap`：一次性数组的第 k 大

```python
import heapq

from collections import Counter

def kth_largest_heap(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('invalid rank')
    heap = []
    for x in nums:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return heap[0]
```

- 开头的 `import` 与 `Counter` 同时服务三个函数。
- `if not 1<=k<=len(nums): raise ValueError` 是输入守卫：k 至少是 1、至多是数组长度，否则"第 k 大"这个词根本没有意义，直接报错比返回垃圾值负责。
- `heap=[]` 起一个空堆。扫描每个数 x：`len(heap)<k` 说明"编制没满"，无条件收人（`heappush`）；编制满了才看 `x>heap[0]`——x 必须严格大于当前门槛才能顶替，`heapreplace` 一步完成"弹旧堆顶、放 x"。
- 全部扫完后 `return heap[0]`：k 个名额里的最小值，正是全局第 k 大。

### 2. `top_k_frequent`：频率 Top-K

```python
def top_k_frequent(nums, k):
    counts = Counter(nums)
    if not 0 <= k <= len(counts):
        raise ValueError('invalid number of distinct keys')
    return heapq.nlargest(k, counts, key=lambda x: (counts[x], -x))
```

- `counts=Counter(nums)` 一行完成计数，比如 `[1,1,1,2,2,3]` 变成 `{1:3, 2:2, 3:1}`。
- 这里的守卫允许 `k=0`（一个都不要，返回空列表），上限换成了 `len(counts)`——不同数值的个数，因为频率榜的单位是"不同的数"，不是元素个数。
- `heapq.nlargest(k, counts, key=...)` 是标准库的"取按 key 排序后最大的 k 个"。注意它在字典上迭代拿到的是键（数值本身）。
- 关键在 `key=lambda x:(counts[x],-x)`：先按频次 `counts[x]` 从大到小；频次并列时按 `-x` 大者优先，而 `-x` 大就是 x 小，所以实现了"并列取数值较小者"的稳定约定。这个元组比较正是第一节说的"并列规则必须自己定义"的落地。

### 3. `KthLargest`：流式版本

```python
class KthLargest:

    def __init__(self, k, nums):
        if k < 1:
            raise ValueError('positive k required')
        self.k = k
        self.heap = []
        for x in nums:
            self.add(x)

    def add(self, value):
        heapq.heappush(self.heap, value)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]
```

- `__init__` 里 `if k<1` 挡住非正的 k（k=0 时"第 0 大"无意义）；初始数组通过循环 `self.add(x)` 喂进去，复用同一个逻辑，避免初始化和增量两套代码。
- `add` 三步：先把新值无条件 `heappush` 进去；`if len(self.heap)>self.k` 发现超员就 `heappop` 弹掉最小值——注意"先加后弹"和 `heapreplace` 的"先弹后加"效果等价，都保证堆里留下最大的 k 个（或不足 k 个时的全体）；最后 `return self.heap[0]` 就是当前第 k 大。
- 一个边界：流中元素不足 k 个时，堆顶是"现有全部数的最小值"。notebook 的正确性一节特别声明：此时返回值只是"当前最小值"，本章不把它吹成严格意义的"第 k 大"，因为第 k 大根本还不存在。

## 5. 分段实现：按顺序运行

**本章接口：** `kth_largest_heap(nums, k)`、`top_k_frequent(nums, k)`、`KthLargest`

### 导入本章使用的库

In [1]:
import heapq
from collections import Counter

### kth_largest_heap

In [2]:
def kth_largest_heap(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('invalid rank')
    heap = []
    for x in nums:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return heap[0]

### top_k_frequent

In [3]:
def top_k_frequent(nums, k):
    counts = Counter(nums)
    if not 0 <= k <= len(counts):
        raise ValueError('invalid number of distinct keys')
    return heapq.nlargest(k, counts, key=lambda x: (counts[x], -x))

### KthLargest

In [4]:
class KthLargest:

    def __init__(self, k, nums):
        if k < 1:
            raise ValueError('positive k required')
        self.k = k
        self.heap = []
        for x in nums:
            self.add(x)

    def add(self, value):
        heapq.heappush(self.heap, value)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

stream=KthLargest(3,[4,5,8,2]); rows=[]
for x in [3,5,10,9,4]:
    result=stream.add(x); rows.append((x,sorted(stream.heap),result))
show_table(['新值','保留的Top3','第3大'],rows)

新值,保留的Top3,第3大
3,"[4, 5, 8]",4
5,"[5, 5, 8]",5
10,"[5, 8, 10]",5
9,"[8, 9, 10]",8
4,"[8, 9, 10]",8


## 7. 正确性、适用条件与复杂度

对输入前缀归纳：堆保存其最大的min(k,t)个元素；插入后删除最小者仍保留该集合。流长度不足k时本章返回当前最小值，并不称它为已存在的“第k大”。

**代价：** Top-k扫描O(n log k)、O(k)空间；频次版O(n+u log k)、O(u)空间。

### 展开理解

为什么扫一遍就能保证堆里永远是"到目前为止最大的 k 个"？我们顺着流的顺序想：假设处理完前 t 个数后，堆里恰好是这 t 个数中最大的 min(k,t) 个。第 t+1 个数 x 到来时只有两种情况：x 比堆顶大，它替换堆顶后，堆里仍是前 t+1 个数中最大的 k 个（原来的门槛被 x 顶掉，x 排进了前 k，最旧的门槛出局，恰好正确）；x 不比堆顶大，它进不了前 k，扔掉也不影响。初始时堆是空的，显然成立，于是这个性质一步一步传到流的结尾。注意堆的大小永不超过 k，所以内存被钉死在 k 上。

复杂度：每个数至多一次进堆、一次出堆，每次堆操作 O(log k)，所以扫描 n 个数是 O(n log k)、额外空间 O(k)。举例：n=10⁶、k=10 时，log₂10≈3.3，总共约 330 万次比较，一瞬间跑完；对比全排序 n log n 约 2000 万次比较。频率版是"O(n) 计数 + 在 u 个不同键里选 k 个"，即 O(n + u log k)、O(u) 空间；当所有数都不同时 u=n，退化成 O(n log k)，仍然便宜。

## 8. 单元测试：每个用例在检查什么

```python
assert kth_largest_heap([3, 2, 1, 5, 6, 4], 2) == 5
```
正常用例：6 个数里第 2 大。降序是 6,5,4,3,2,1，第 2 位是 5，验证门槛堆给出的名次正确。

```python
assert set(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == {1, 2}
```
正常用例：频次 3、2、1 取前两名得 1 和 2。注意这里特意用 `set(...)` 比较，因为频率题的输出顺序不应成为契约的一部分。

```python
stream = KthLargest(3, [4, 5, 8, 2])

assert [stream.add(x) for x in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]
```
流式用例：逐次 add 并收集返回值，正好是第三节手算表最后一列 [4,5,5,8,8]。这条测的是"增量维护的结果每一步都正确"，而不是只测最后一步。

```python
rng = Random(58)

for k in range(1, 8):
    seen = []
    stream = KthLargest(k, [])
    for _ in range(40):
        x = rng.randrange(20)
        seen.append(x)
        answer = stream.add(x)
        assert answer == sorted(seen)[-min(k, len(seen))]
```
随机对照测试：k 从 1 取到 7，每轮往流里随机扔 40 个 0~19 的数，每一步都和暴力参照——"把见过的数排序后取倒数第 min(k, 已见个数) 个"——比对。`min(k,len(seen))` 正是照顾"流还没攒够 k 个数"的边界。k=1（只要最大值）和 k=7（大于随机数值种类数 20 的情形之外又接近流长的一半）这些极端形状都被覆盖了。

In [6]:
assert kth_largest_heap([3, 2, 1, 5, 6, 4], 2) == 5

assert set(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == {1, 2}

stream = KthLargest(3, [4, 5, 8, 2])

assert [stream.add(x) for x in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]

from random import Random

rng = Random(58)

for k in range(1, 8):
    seen = []
    stream = KthLargest(k, [])
    for _ in range(40):
        x = rng.randrange(20)
        seen.append(x)
        answer = stream.add(x)
        assert answer == sorted(seen)[-min(k, len(seen))]

print('N058: 所有本章断言通过')

N058: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 比较排序、快选与堆的时间空间。

**练习 2：** 定义频率并列时的合法输出判定。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- 练习 1（比较排序、快选与堆）：三种做法各写一行账。全排序：O(n log n) 时间、可以原地；快速选择（quickselect）：平均 O(n)、最坏 O(n²)、需要能随机访问整个数组；门槛堆：O(n log k)、O(k) 空间、唯一能应付流式输入的。提示：手算例子可以直接用本章的 `[3,2,1,5,6,4], k=2`，分别数一数三种方法各自做了多少次比较。
- 练习 2（定义频率并列时的合法输出判定）：当两个数频次相同且正好卡在第 k 名边界时，任何一个都可能合法。提示：合法输出集合可以描述为"任取一个键集合 S，|S|=k，且 S 中每个键的频次都不小于 S 外任何键的频次"；再写一个 `is_valid_output(nums, k, answer)` 的判定函数，用它而不是 `==` 去验收各种并列输入。本章代码取"数值较小者"只是这个合法集合中的一种稳定选择。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
import heapq

from collections import Counter

def kth_largest_heap(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('invalid rank')
    heap = []
    for x in nums:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return heap[0]

def top_k_frequent(nums, k):
    counts = Counter(nums)
    if not 0 <= k <= len(counts):
        raise ValueError('invalid number of distinct keys')
    return heapq.nlargest(k, counts, key=lambda x: (counts[x], -x))

class KthLargest:

    def __init__(self, k, nums):
        if k < 1:
            raise ValueError('positive k required')
        self.k = k
        self.heap = []
        for x in nums:
            self.add(x)

    def add(self, value):
        heapq.heappush(self.heap, value)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]

# 示例与回归测试
assert kth_largest_heap([3, 2, 1, 5, 6, 4], 2) == 5

assert set(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == {1, 2}

stream = KthLargest(3, [4, 5, 8, 2])

assert [stream.add(x) for x in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]

from random import Random

rng = Random(58)

for k in range(1, 8):
    seen = []
    stream = KthLargest(k, [])
    for _ in range(40):
        x = rng.randrange(20)
        seen.append(x)
        answer = stream.add(x)
        assert answer == sorted(seen)[-min(k, len(seen))]

print('N058: 所有本章断言通过')

N058: 所有本章断言通过


## 11. 代表题与迁移

- **215. Kth Largest Element in an Array（数组中的第 K 个最大元素）**：本章 `kth_largest_heap` 的原题，练的是"大小 k 的小根堆做门槛"。
- **347. Top K Frequent Elements（前 K 个高频元素）**：对应 `top_k_frequent`，练的是"Counter 计数 + 按频率 Top-K + 自定义并列规则"。
- **703. Kth Largest Element in a Stream（数据流中的第 K 大元素）**：对应 `KthLargest` 类，练的是"堆的增量维护：来一个、弹一个、堆顶即答案"。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。